# Dynamic Programming

Notes and runnable examples on dynamic programming (DP) — the technique for problems with **overlapping subproblems** and **optimal substructure**. It's exactly the memoization idea from the recursion notebook, made systematic: solve each subproblem once, reuse it everywhere. We'll do it both ways (top-down memo, bottom-up table) and work the classic grid DPs — coin change, LCS, edit distance, knapsack.

**Contents**

1. **What DP is** — the two ingredients
2. **Top-down vs bottom-up** — coin change
3. **A 2D DP** — longest common subsequence
4. **Edit distance** (Levenshtein)
5. **0/1 knapsack** — and shrinking the table
6. **More classics** — LIS, subset sum, partition
7. **Recognizing & designing a DP**
8. **Complexity cheat-sheet**

## 1. What DP is — the two ingredients

Dynamic programming applies when a problem has both:

1. **Optimal substructure** — the best solution is built from best solutions to subproblems.
2. **Overlapping subproblems** — those subproblems recur many times, so caching pays off.

That second property is the whole point. The naive recursive Fibonacci from the previous notebook had optimal substructure *and* overlapping subproblems — which is why memoizing it collapsed exponential work to linear. DP is just that idea applied deliberately, in one of two styles:

- **Top-down (memoization):** write the natural recursion, then cache results (`functools.cache`). Lazy — only solves the subproblems you actually reach.
- **Bottom-up (tabulation):** fill a table from the smallest subproblems upward, in dependency order. No recursion, no stack limit, and usually easier to **shrink the memory** afterward.

Same recurrence, two implementations — reach for whichever maps more naturally onto the problem.

## 2. Top-down vs bottom-up — coin change

The **coin change** problem: given coin denominations and a target amount, what's the fewest coins that sum to it? The recurrence is `best(amount) = 1 + min(best(amount - c) for each coin c)`, with `best(0) = 0`. (Greedy fails here — with coins `[1, 3, 4]` and amount 6 it grabs 4+1+1 = 3 coins, but 3+3 = **2** is optimal. DP weighs all options.)

Here it is both ways, giving the same answer:

In [1]:
from functools import cache

def coin_change_topdown(coins, amount):
    @cache                                       # memoize each subproblem
    def best(rem):
        if rem == 0:
            return 0
        if rem < 0:
            return float("inf")                  # this path can't reach the target
        return min(best(rem - c) + 1 for c in coins)
    result = best(amount)
    return result if result != float("inf") else -1

def coin_change_bottomup(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount                    # dp[a] = fewest coins to make a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != INF else -1

coins = [1, 3, 4]
print("top-down :", coin_change_topdown(coins, 6))
print("bottom-up:", coin_change_bottomup(coins, 6))
print("(greedy would pick 4+1+1 = 3 coins; DP finds 3+3 = 2)")


top-down : 2
bottom-up: 2
(greedy would pick 4+1+1 = 3 coins; DP finds 3+3 = 2)


## 3. A 2D DP — longest common subsequence

Many DPs live on a **grid**, with one input along each axis. The **longest common subsequence (LCS)** of two strings is the canonical example — it underlies `diff` and DNA alignment. Build a table where `dp[i][j]` = LCS length of the first `i` chars of `a` and first `j` of `b`:

- if `a[i-1] == b[j-1]` the characters match → extend the diagonal: `dp[i-1][j-1] + 1`
- otherwise drop a character from one side → `max(dp[i-1][j], dp[i][j-1])`

The answer is the bottom-right corner; walking *back* from there recovers the actual subsequence:

In [2]:
def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1               # match: extend the diagonal
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])    # skip a char from a or b
    return dp

a, b = "ABCBDAB", "BDCAB"
dp = lcs(a, b)

print("       " + "".join(f"{c:>3}" for c in ("." + b)))      # '.' = the empty-prefix column
for i, row in enumerate(dp):
    label = "." if i == 0 else a[i - 1]
    print(f"   {label}  " + "".join(f"{x:>3}" for x in row))

# reconstruct one LCS by walking back from the corner
i, j, out = len(a), len(b), []
while i > 0 and j > 0:
    if a[i - 1] == b[j - 1]:
        out.append(a[i - 1]); i -= 1; j -= 1
    elif dp[i - 1][j] >= dp[i][j - 1]:
        i -= 1
    else:
        j -= 1
print("LCS length:", dp[-1][-1], "| one such subsequence:", "".join(reversed(out)))


         .  B  D  C  A  B
   .    0  0  0  0  0  0
   A    0  0  0  0  1  1
   B    0  1  1  1  1  2
   C    0  1  1  2  2  2
   B    0  1  1  2  2  3
   D    0  1  2  2  2  3
   A    0  1  2  2  3  3
   B    0  1  2  2  3  4
LCS length: 4 | one such subsequence: BCAB


## 4. Edit distance (Levenshtein)

**Edit distance** is the minimum number of single-character **insertions, deletions, or substitutions** to turn one string into another — the engine behind spell-checkers and fuzzy matching. Same grid shape as LCS, but each cell takes the cheapest of three moves (plus base cases for turning a string into the empty string):

In [3]:
def edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i                             # delete all i chars of a
    for j in range(n + 1):
        dp[0][j] = j                             # insert all j chars of b
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,                # delete from a
                dp[i][j - 1] + 1,                # insert into a
                dp[i - 1][j - 1] + cost,         # substitute (cost 0 if the chars match)
            )
    return dp[m][n]

print("kitten -> sitting :", edit_distance("kitten", "sitting"))
print("sunday -> saturday:", edit_distance("sunday", "saturday"))


kitten -> sitting : 3
sunday -> saturday: 3


## 5. 0/1 knapsack — and shrinking the table

The **0/1 knapsack**: choose a subset of items (each taken or not) to maximize value without exceeding a weight capacity. The 2D table `dp[i][w]` = best value using the first `i` items within capacity `w`; each item is either skipped or taken:

In [4]:
def knapsack(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]                          # skip item i
            if weights[i - 1] <= w:                          # or take it, if it fits
                dp[i][w] = max(dp[i][w],
                               dp[i - 1][w - weights[i - 1]] + values[i - 1])
    return dp[n][capacity]

weights, values, cap = [1, 3, 4, 5], [1, 4, 5, 7], 7
print("knapsack (2D):", knapsack(weights, values, cap))   # 9 = items of weight 3 (v4) + 4 (v5)


knapsack (2D): 9


Each row of that table only reads the row directly above it — so you don't need to keep them all. A single 1D array suffices if you sweep the capacity **backwards** (which guarantees each item is used at most once). That drops the space from O(n·W) to **O(W)** — a common and important DP optimization:

In [5]:
def knapsack_1d(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(weights)):
        for w in range(capacity, weights[i] - 1, -1):       # backward => each item used once
            dp[w] = max(dp[w], dp[w - weights[i]] + values[i])
    return dp[capacity]

print("knapsack (1D):", knapsack_1d(weights, values, cap), "(same answer, O(W) space)")


knapsack (1D): 9 (same answer, O(W) space)


## 6. More classics — LIS, subset sum, partition

Three more textbook DPs that round out the pattern vocabulary. Each one introduces a state shape you'll meet again: a *single index* with a backward scan (LIS), and *boolean reachability* over a target (subset sum, and its twin, partition). We verify every result against brute force.


**Longest increasing subsequence.** Given a sequence, find the length of the longest strictly increasing subsequence (not necessarily contiguous). The textbook DP defines `L[i]` = length of the longest increasing subsequence *ending at* index `i`; it's `1 + max(L[j])` over every earlier `j` with `nums[j] < nums[i]`. That's two nested loops → **O(n²)**.

There's a faster way. Keep a list `tails` where `tails[k]` is the smallest possible tail of an increasing subsequence of length `k+1`. For each value, find the first tail `>= value` and overwrite it (or append if the value beats every tail). That list stays sorted, so the search is a binary search — `bisect.bisect_left` — giving **O(n log n)**. This is the *patience sorting* idea; `tails` isn't a real subsequence, but its **length** is the answer. (See the **searching** notebook for `bisect` and the sorted-list invariant it relies on.)


In [6]:
import bisect
import random
from itertools import combinations

def lis_dp(nums):                                    # O(n^2): L[i] ends at i
    if not nums:
        return 0
    L = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                L[i] = max(L[i], L[j] + 1)
    return max(L)

def lis_patience(nums):                              # O(n log n): bisect on tails
    tails = []                                        # tails[k] = smallest tail of an LIS of length k+1
    for x in nums:
        k = bisect.bisect_left(tails, x)              # first tail >= x  (strictly increasing)
        if k == len(tails):
            tails.append(x)                           # x extends the longest chain
        else:
            tails[k] = x                              # x is a smaller tail for that length
    return len(tails)

def lis_brute(nums):                                  # check: longest strictly-increasing subseq
    best = 0
    for r in range(len(nums) + 1):
        for combo in combinations(nums, r):
            if all(combo[i] < combo[i + 1] for i in range(len(combo) - 1)):
                best = max(best, r)
    return best

demo = [10, 9, 2, 5, 3, 7, 101, 18]
print("nums      :", demo)
print("LIS length:", lis_patience(demo))             # 4  -> e.g. [2, 3, 7, 18] or [2, 5, 7, 101]

rng = random.Random(0)
for _ in range(300):                                  # both DPs == brute force on small arrays
    arr = [rng.randint(0, 9) for _ in range(rng.randint(0, 8))]
    exp = lis_brute(arr)
    assert lis_dp(arr) == exp, arr
    assert lis_patience(arr) == exp, arr
print("both methods match brute force on 300 random arrays ✓")


nums      : [10, 9, 2, 5, 3, 7, 101, 18]
LIS length: 4
both methods match brute force on 300 random arrays ✓


**Subset sum.** Can any subset of the numbers add up exactly to a target `T`? The state is a boolean `reachable[s]` = "some subset sums to `s`". Start with `reachable[0] = True` (the empty subset), then fold each number in: any sum `s` you could already make, you can now also make `s + num`. Sweeping `s` **backwards** keeps each number used at most once — the same trick as the 1D knapsack above. Cost is **O(n·T)** (pseudo-polynomial in `T`).


In [7]:
def subset_sum(nums, target):                         # O(n*T): boolean reachability
    reachable = [False] * (target + 1)
    reachable[0] = True                               # empty subset sums to 0
    for num in nums:
        for s in range(target, num - 1, -1):          # backward => use each num once
            if reachable[s - num]:
                reachable[s] = True
    return reachable[target]

def subset_sum_brute(nums, target):                   # check: any subset hits the target?
    n = len(nums)
    return any(
        sum(nums[i] for i in range(n) if mask >> i & 1) == target
        for mask in range(1 << n)
    )

print("subset of [3,34,4,12,5,2] summing to 9 ?", subset_sum([3, 34, 4, 12, 5, 2], 9))   # 4+5
print("                            ... to 30 ?", subset_sum([3, 34, 4, 12, 5, 2], 30))

rng = random.Random(1)
for _ in range(300):
    arr = [rng.randint(0, 8) for _ in range(rng.randint(0, 9))]
    T = rng.randint(0, 20)
    assert subset_sum(arr, T) == subset_sum_brute(arr, T), (arr, T)
print("subset_sum matches brute force on 300 random cases ✓")


subset of [3,34,4,12,5,2] summing to 9 ? True
                            ... to 30 ? False
subset_sum matches brute force on 300 random cases ✓


**Partition.** Can the array be split into two groups with equal sums? If the total is odd, no — you can't halve an odd number. Otherwise it's *exactly* subset sum to `total // 2`: any subset reaching half the total leaves its complement reaching the other half. So partition is a one-line wrapper, which is the point — recognizing a problem as a known DP in disguise is half the battle.


In [8]:
def can_partition(nums):                              # equal-sum halves == subset_sum(total/2)
    total = sum(nums)
    if total % 2:
        return False                                  # odd total can't be split evenly
    return subset_sum(nums, total // 2)

def can_partition_brute(nums):                        # check: any subset == its complement's sum
    total = sum(nums)
    n = len(nums)
    return any(
        sum(nums[i] for i in range(n) if mask >> i & 1) * 2 == total
        for mask in range(1 << n)
    )

print("[1,5,11,5] partitions evenly ?", can_partition([1, 5, 11, 5]))   # 11 == 1+5+5
print("[1,2,3,5]  partitions evenly ?", can_partition([1, 2, 3, 5]))    # total 11 is odd

rng = random.Random(2)
for _ in range(300):
    arr = [rng.randint(0, 8) for _ in range(rng.randint(0, 9))]
    assert can_partition(arr) == can_partition_brute(arr), arr
print("can_partition matches brute force on 300 random cases ✓")


[1,5,11,5] partitions evenly ? True
[1,2,3,5]  partitions evenly ? False
can_partition matches brute force on 300 random cases ✓


These three are the gateway to a whole family of harder DPs: LIS generalizes into longest-chain and box-stacking problems, while subset sum and partition are the entry point to the *knapsack family* — bounded, unbounded, and multi-dimensional variants. For the optimizations that tame them (monotonic-queue, convex-hull, divide-and-conquer, and bitset speedups), see the **advanced dynamic programming** notebook.


## 7. Recognizing & designing a DP

A reliable recipe when you suspect DP:

1. **Spot the signs** — you're optimizing (min / max / count) over a sequence of choices, and a brute-force recursion would re-solve the same subproblems.
2. **Define the state** — what parameters identify a subproblem? (an index, a remaining amount, a pair of positions). This becomes your cache key, or your table's dimensions.
3. **Write the recurrence** — express a state's answer in terms of *smaller* states, and nail down the base cases.
4. **Pick a direction** — top-down (`@cache` the recurrence — fastest to write) or bottom-up (fill a table in dependency order — no recursion limit, easier to shrink).
5. **Optimize space** — if each layer only depends on the previous one, roll the table down to 1D.

The hardest step is almost always **#2, defining the state**. Once the state and recurrence are right, the code is mechanical.

## 8. Complexity cheat-sheet

| Problem | State | Time | Space (naive → rolled) |
|---|---|:---:|:---:|
| Fibonacci | `n` | O(n) | O(n) → O(1) |
| Coin change | amount | O(amount × #coins) | O(amount) |
| Longest common subsequence | `(i, j)` | O(m·n) | O(m·n) → O(min(m, n)) |
| Edit distance | `(i, j)` | O(m·n) | O(m·n) → O(min(m, n)) |
| 0/1 knapsack | `(item, capacity)` | O(n·W) | O(n·W) → O(W) |
| Longest increasing subseq. | `i` / patience | O(n²) or O(n log n) | O(n) |

<sub>W = capacity. Knapsack is *pseudo*-polynomial: its cost scales with the numeric capacity W, not just the input size. Top-down memoization has the same time complexity as bottom-up but adds call-stack + hashing overhead.</sub>